# Global RF
- Preprocess data
- Compare linear and non linear model
- Explore Clustering of residuals
- Spatial Crosvalidation

In [ ]:
import pathlib

import esda
import geopandas as gpd
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shapely
from libpysal import graph
from sklearn import ensemble, metrics, model_selection
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

## Preprocess the data
Remove unnecessary column, devide the data by total population and standardize it.

In [ ]:
def process_file(path, path_total):
    cols_no_division = [
        "Hustota obyvatel na obytnou plochu",
        "Počet obyvatel na dům",
        "Počet obyvatel na byt",
    ]

    # Load total population data
    total = pd.read_csv(path_total, dtype={"nadzsjd": str}, index_col=0)

    # Load main data
    data = gpd.read_parquet(path)

    # Merge data
    data_total = data.join(total)

    # Drop unnecessary columns
    data_relative = data_total.drop(
        columns=[
            "NUTS_2",
            "naz_oblast",
            "NUTS_3",
            "naz_kraj",
            "kod_okres",
            "naz_okres",
            "naz_orp",
            "kod_obec",
            "naz_obec",
            "kod_mco",
            "nazev_mco",
        ]
    )

    # Convert columns (except 'geometry') to float
    cols_numeric = data_relative.columns.drop(["geometry", "kod_orp"])
    data_relative[cols_numeric] = data_relative[cols_numeric].astype(float)

    # Normalize by total population (except certain columns)
    population = data_relative["Obyvatelstvo celkem"].replace(
        0, np.nan
    )  # avoid division by zero
    cols_to_normalize = [
        col
        for col in cols_numeric
        if col not in cols_no_division + ["Obyvatelstvo celkem"]
    ]
    data_relative[cols_to_normalize] = data_relative[cols_to_normalize].div(
        population, axis=0
    )

    # Drop rows with NaNs after division
    data_relative = data_relative.dropna(subset=cols_to_normalize)

    # Clip extremely large or small values
    data_relative[cols_to_normalize] = data_relative[cols_to_normalize].clip(-1e6, 1e6)
    data_relative[cols_no_division] = data_relative[cols_no_division].clip(-1e6, 1e6)

    # Replace any remaining infinities with NaN and drop them
    data_relative.replace([np.inf, -np.inf], np.nan, inplace=True)
    data_relative.dropna(subset=cols_to_normalize + cols_no_division, inplace=True)

    # Scale columns
    #    scaler = StandardScaler()
    #   data_relative[cols_to_normalize] = scaler.fit_transform(data_relative[cols_to_normalize])
    #   data_relative[cols_no_division] = scaler.fit_transform(data_relative[cols_no_division])

    return data_relative

In [ ]:
path = "/data/uscuni-restricted/04_spatial_census/_merged_census_2021.parquet"
path_total = "/data/uscuni-restricted/04_spatial_census/total.csv"

In [ ]:
data_relative = process_file(path, path_total)

In [ ]:
data_r = data_relative[data_relative.columns.drop(["Obyvatelstvo celkem", "kod_orp"])]
data_r.to_parquet(
    "/data/uscuni-restricted/04_spatial_census/_merged_census_2021_relative_scaled.parquet"
)

## Assign cluster label


In [ ]:
clusters = pd.read_csv(
    "/data/uscuni-restricted/04_spatial_census/cluster_assignment_v10.csv",
    dtype={"kod_nadzsj_d": str},
)
cluster_mapping = pd.read_parquet(
    "/data/uscuni-ulce/processed_data/clusters/cluster_mapping_v10.pq"
)
data = data_relative.merge(clusters, left_on="nadzsjd", right_on="kod_nadzsj_d")
variables = data.columns.drop(
    [
        "geometry",
        "kod_nadzsj_d",
        "final_without_noise",
        "kod_orp",
        "Obyvatelstvo celkem",
    ]
)

data["Cluster"] = data["final_without_noise"].map(cluster_mapping[3])

In [ ]:
data["Cluster"].unique()

In [ ]:
pca = [
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání: VOŠ a konzervatoř - celkem",
    "Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: zemědělství, lesnictví ,rybářství - celkem",
    "Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: průmysl - celkem",
    "Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: stavebnictví - celkem",
    "Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: doprava a skladování - celkem",
    "Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: ubytování, stravování a pohostinství - celkem",
    "Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: veřejná správa a obrana, povinné sociální zabezpečení - celkem",
    "Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: vzdělávání - celkem",
    "Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: zdravotní a sociální péče - celkem",
    "Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: nezjištěno - celkem",
    "Zaměstnaní - Zaměstnanci v ozbrojených silách",
    "Zaměstnaní - Pracovníci ve službách a prodeji",
    "Zaměstnaní - Kvalifikovaní pracovníci v zemědělství,lesnictví a rybářství",
    "Zaměstnaní - Řemeslníci a opraváři",
    "Obyvatelstvo - zaměstnaní - postavení v zaměstnání: zaměstnavatelé - celkem",
    "Obyvatelstvo - zaměstnaní - postavení v zaměstnání: osoby pracující na vlastní účet - celkem",
    "Obyvatelstvo - zaměstnaní - postavení v zaměstnání: nezjištěno - celkem",
    "Hospodařící domácnosti v bytech - počet členů domácnosti: 5 a více",
    "Počet osob v bytech celkem  s právním důvodem užívání: v osobním vlastnictví",
    "Počet obyvatel na byt",
    "Počet osob v domech celkem s vlastnictvím:   obec, stát",
    "Počet osob v domech celkem s vlastnictvím:   bytové družstvo",
    "Počet obyvatel na dům",
    "Obyvatelstvo - věk: 85 a více  - celkem",
    "Obyvatelstvo - věk: 15 - 24  - celkem",
    "Obyvatelstvo - věk: 35 - 44  - celkem",
    "Obyvatelstvo - věk: 45 - 54  - celkem",
    "Obyvatelstvo - věk: 55 - 64  - celkem",
    "Obyvatelstvo - ekon. aktivita: osoby na mateřské dovolené - celkem",
    "Obyvatelstvo - ekon. aktivita: ostatní s vlastním zdrojem obživy - celkem",
    "Obyvatelstvo - ekon. aktivita: osoby na rodičovské dovolené - celkem",
    "Obyvatelstvo - státní občanství: Slovenská republika - celkem",
    "Obyvatelstvo - státní občanství: země EU mimo ČR - celkem",
    "Obyvatelstvo - bez státního občanství - celkem",
    "Obyvatelstvo - státní občanství: nezjištěno - celkem",
    "Obyvatelstvo - náboženská víra: věřící - nehlásící se k církvi, náboženské společnosti nebo směru - celkem",
    "Obyvatelstvo - náboženská víra: věřící - hlásící se k církvi, náboženské společnosti nebo směru - celkem",
    "Obyvatelstvo - náboženská víra: bez náboženské víry - celkem",
    "Obyvatelstvo - náboženská víra: neuvedeno - celkem",
    "Obyvatelstvo - rodinný stav: nezjištěno - celkem",
]

## Classification


In [ ]:
# Assign independent variables and the target
independent = data[pca]
target = data["Cluster"]

In [ ]:
# Plot original data
ax = data.plot(target, legend=True, figsize=(9, 9), markersize=0.1, categorical=True)
ax.set_axis_off()

In [ ]:
# Split data
X_train, X_test, y_train, y_test = model_selection.train_test_split(
    independent,
    target,
    test_size=0.2,
    random_state=42,
)

### Linear model


In [ ]:
linear_model = LogisticRegression(
    n_jobs=-1, max_iter=1000, solver="saga", random_state=42, class_weight="balanced"
)
linear_model.fit(X_train, y_train)
linear_model.score(X_train, y_train), linear_model.score(X_test, y_test)

### Non-Linear model


In [ ]:
rf_model = RandomForestClassifier(random_state=42, n_jobs=-1, class_weight="balanced")
rf_model.fit(X_train, y_train)
rf_model.score(X_train, y_train), rf_model.score(X_test, y_test)

#### Parameter tuning


In [ ]:
np.mean([estimator.tree_.max_depth for estimator in rf_model.estimators_])

In [ ]:
rf_model = ensemble.RandomForestClassifier(
    min_samples_split=10,
    min_samples_leaf=5,
    max_depth=15,
    n_jobs=-1,
    random_state=42,
    n_estimators=300,
    max_features="log2",
    class_weight="balanced",
)
rf_model.fit(X_train, y_train)

In [ ]:
rf_model.score(X_train, y_train), rf_model.score(X_test, y_test)

In [ ]:
pred = rf_model.predict(X_test)
pred

In [ ]:
proba = rf_model.predict_proba(X_test)
pd.DataFrame(proba, columns=rf_model.classes_, index=X_test.index)

In [ ]:
accuracy = metrics.accuracy_score(pred, y_test)
kappa = metrics.cohen_kappa_score(pred, y_test)

summary = f"""\
Evaluation metrics
==================
Basic model:
  Accuracy: {round(accuracy, 3)}
  Kappa:    {round(kappa, 3)}
"""

print(summary)

In [ ]:
predicted = model_selection.cross_val_predict(
    rf_model, independent, target, cv=4, n_jobs=-1
)

In [ ]:
predicted = model_selection.cross_val_predict(
    rf_model, independent, target, cv=4, n_jobs=-1
)

ax = data.plot(predicted, legend=True, figsize=(9, 9), markersize=0.1, categorical=True)
ax.set_axis_off()

#### Probabilities


In [ ]:
probas = model_selection.cross_val_predict(
    rf_model, independent, target, cv=4, n_jobs=-1, method="predict_proba"
)

In [ ]:
probas = pd.DataFrame(probas, columns=rf_model.classes_, index=independent.index)
probas

In [ ]:
probas["max_value"] = probas.max(axis=1)
probas["max_label"] = probas.idxmax(axis=1)

probas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

data.plot(
    column=probas["max_value"],
    legend=True,
    markersize=0.1,
    ax=axes[0],
    vmin=0.2,
    vmax=0.8,
    cmap="YlGnBu",
)
axes[0].set_title("Max Probability")
axes[0].set_axis_off()

data.plot(column=probas["max_label"], legend=True, markersize=0.1, ax=axes[1])
axes[1].set_title("Predicted Class (argmax)")
axes[1].set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(8, 2, figsize=(12, 32))


prob_columns = probas.columns[:8]

for i in range(8):
    subset = data[target == i + 1]

    subset.plot(
        column=target.name, legend=True, markersize=0.1, categorical=True, ax=axes[i, 0]
    )

    axes[i, 0].set_axis_off()

    data.plot(
        column=probas[prob_columns[i]],
        legend=True,
        markersize=0.1,
        cmap="YlGnBu",
        vmin=0,
        vmax=0.8,
        ax=axes[i, 1],
    )

    axes[i, 1].set_axis_off()

plt.tight_layout()
plt.show()

### Residual exploration

In [ ]:
ax = data.plot(
    predicted == target,
    categorical=True,
    figsize=(9, 9),
    markersize=0.1,
    cmap="bwr_r",
    legend=True,
)
ax.set_axis_off()

In [ ]:
data["error"] = predicted == target
data["error"] = data["error"].astype(int).astype(float).values

In [ ]:
contiguity = graph.Graph.build_knn(data.centroid, k=8)
contiguity_b = contiguity.transform("b")

### Join Counts

In [ ]:
jc = esda.Join_Counts(data["error"], contiguity_b)


# Results
print("BB count:", jc.bb)  # 1–1 joins
print("WW count:", jc.ww)  # 0–0 joins
print("BW count:", jc.bw)
print("mean BB:", jc.mean_bb)
print("p-value:", jc.p_sim_bb)

### Local Join Counts

In [ ]:
ljc = esda.Join_Counts_Local(connectivity=contiguity).fit(data["error"])

In [ ]:
data.plot(ljc.LJC, cmap="bwr", legend=True)

In [ ]:
data["ljc"] = ljc.p_sim < 0.05

In [ ]:
data.plot(data["ljc"], cmap="bwr", legend=True)

#### Getirs Ord

In [ ]:
g_local = esda.G_Local(data["error"], contiguity_b, star=True, permutations=999)
data["G_sig"] = g_local.p_sim < 0.05

In [ ]:
data.plot(data["G_sig"], cmap="bwr", legend=True)

In [ ]:
data["spot_type"] = "Insignificant"

data.loc[(g_local.p_sim < 0.05) & (g_local.Zs > 0), "spot_type"] = "Hot Spot"
data.loc[(g_local.p_sim < 0.05) & (g_local.Zs < 0), "spot_type"] = "Cold Spot"

data["spot_type"].value_counts()

In [ ]:
f, ax = plt.subplots()
data.loc[data["spot_type"] == "Insignificant"].plot(ax=ax, color="lightgrey")
data.loc[data["spot_type"] == "Hot Spot"].plot(ax=ax, color="#d7191c")
data.loc[data["spot_type"] == "Cold Spot"].plot(ax=ax, color="#2c7bb6")

#### Local Moran I

In [ ]:
# Calculate Local Moran's I
lisa = esda.moran.Moran_Local(data["error"], contiguity_b, permutations=999)

In [ ]:
lisa.plot(
    data,
    crit_value=0.05,
)

### Spatial Crossvalidation

In [ ]:
gkf = model_selection.StratifiedGroupKFold(n_splits=5)
splits = gkf.split(
    independent,
    target,
    groups=data.kod_orp,
)

In [ ]:
split_label = np.empty(len(data), dtype=float)
split_label

In [ ]:
for i, (_train_idx, test_idx) in enumerate(splits):
    split_label[test_idx] = i
data["split"] = split_label

In [ ]:
ax = data.plot("split", categorical=True, figsize=(9, 9), markersize=0.1, legend=True)
data.dissolve("kod_orp").convex_hull.boundary.plot(
    ax=ax, color="k", linewidth=0.5, markersize=0
)
ax.set_axis_off()

In [ ]:
train = data["split"] != 0
X_train = independent.loc[train]
y_train = data["Cluster"].loc[train]

test = data["split"] == 0
X_test = independent.loc[test]
y_test = data["Cluster"].loc[test]

In [ ]:
rf_spatial_cv = rf_model
rf_spatial_cv.fit(X_train, y_train)

In [ ]:
rf_spatial_cv.score(X_train, y_train), rf_spatial_cv.score(X_test, y_test)

In [ ]:
pred = rf_spatial_cv.predict(X_test)

accuracy_spatial_cv = metrics.accuracy_score(pred, y_test)
kappa_spatial_cv = metrics.cohen_kappa_score(pred, y_test)

summary += f"""\
Basic model with spatial cross-validation:
  Accuracy: {round(accuracy_spatial_cv, 3)}
  Kappa:    {round(kappa_spatial_cv, 3)}
"""

print(summary)

### Spatial lag

In [ ]:
contiguity = graph.Graph.build_knn(data.centroid, k=8)
contiguity = contiguity.transform("b")

In [ ]:
lagged_variables = []
for var in independent:
    data[f"{var}_lag"] = contiguity.lag(data[var])
    lagged_variables.append(f"{var}_lag")
data.head(2)

In [ ]:
independent_lag = data[pca + lagged_variables]
independent.head(2)

In [ ]:
X_train = independent_lag.loc[train]
y_train = target.loc[train]

X_test = independent_lag.loc[test]
y_test = target.loc[test]

In [ ]:
rf_lag = ensemble.RandomForestClassifier(
    random_state=42, n_jobs=-1, class_weight="balanced"
)
rf_lag.fit(X_train, y_train)
rf_lag.score(X_train, y_train), rf_lag.score(X_test, y_test)

In [ ]:
np.mean([estimator.tree_.max_depth for estimator in rf_lag.estimators_])

In [ ]:
rf_lag = ensemble.RandomForestClassifier(
    min_samples_split=10,
    min_samples_leaf=5,
    max_depth=15,
    n_jobs=-1,
    random_state=42,
    n_estimators=300,
    max_features="log2",
    class_weight="balanced",
)
rf_lag.fit(X_train, y_train)

In [ ]:
rf_lag.score(X_train, y_train), rf_lag.score(X_test, y_test)

In [ ]:
pred = rf_lag.predict(X_test)

accuracy_lag = metrics.accuracy_score(pred, y_test)
kappa_lag = metrics.cohen_kappa_score(pred, y_test)

summary += f"""\
Spatial dependence - lagged model (spatial CV):
  Accuracy: {round(accuracy_lag, 3)}
  Kappa:    {round(kappa_lag, 3)}
"""
print(summary)

In [ ]:
gkf = model_selection.StratifiedGroupKFold(n_splits=10)
splits = gkf.split(
    independent_lag,
    target,
    groups=data.kod_orp,
)
split_label = np.empty(len(data), dtype=float)
for i, (_train_idx, test_idx) in enumerate(splits):
    split_label[test_idx] = i
data["split"] = split_label

In [ ]:
train = data["split"] != 0
X_train = independent_lag.loc[train]
y_train = data["Cluster"].loc[train]

test = data["split"] == 0
X_test = independent_lag.loc[test]
y_test = data["Cluster"].loc[test]

In [ ]:
rf_spatial_cv = rf_lag
rf_spatial_cv.fit(X_train, y_train)

In [ ]:
pred = rf_spatial_cv.predict(X_test)

accuracy_lag = metrics.accuracy_score(pred, y_test)
kappa_lag = metrics.cohen_kappa_score(pred, y_test)

summary += f"""\
Spatial dependence - lagged model (spatial CV):
  Accuracy: {round(accuracy_lag, 3)}
  Kappa:    {round(kappa_lag, 3)}
"""
print(summary)

In [ ]:
X_train

### Feature importance

In [ ]:
feat_importances = pd.Series(
    rf_lag.feature_importances_, index=X_train.columns
).sort_values()
plt.figure(figsize=(5, 20))
plt.axvline(x=0.01, color="red", linestyle="--", linewidth=1)


feat_importances.plot(kind="barh")